In [ ]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\Harin\OneDrive\Documents\Desktop\Sem_5\PR\p_1.csv",
    header=None
)
print(df.shape)
print(df.head())

(376417, 54)
     0   1      2     3        4        5        6        7        8   \
0  8.38   0  104.0  30.0  2.37223  8.60074  3.51048  2.43954  8.76165   
1  8.39   0    NaN  30.0  2.18837  8.56560  3.66179  2.39494  8.55081   
2  8.40   0    NaN  30.0  2.37357  8.60107  3.54898  2.30514  8.53644   
3  8.41   0    NaN  30.0  2.07473  8.52853  3.66021  2.33528  8.53622   
4  8.42   0    NaN  30.0  2.22936  8.83122  3.70000  2.23055  8.59741   

        9   ...        44        45        46       47       48       49   50  \
0  3.35465  ...  0.008300  0.009250 -0.017580 -61.1888 -38.9599 -58.1438  1.0   
1  3.64207  ... -0.006577 -0.004638  0.000368 -59.8479 -38.8919 -58.5253  1.0   
2  3.73280  ...  0.003014  0.000148  0.022495 -60.7361 -39.4138 -58.3999  1.0   
3  3.73277  ...  0.003175 -0.020301  0.011275 -60.4091 -38.7635 -58.3956  1.0   
4  3.76295  ...  0.012698 -0.014303 -0.002823 -61.5199 -39.3879 -58.2694  1.0   

    51   52   53  
0  0.0  0.0  0.0  
1  0.0  0.0  0.0  
2  0

In [13]:
import numpy as np
import pandas as pd

from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split


# ============================================================
# 1. LOAD PAMAP2 DATA
# ===========================================================

# ============================================================
# 2. IDENTIFY LABEL AND FEATURES
# ============================================================

# PAMAP2:
# column 0 = timestamp
# column 1 = activity ID
# columns 2 onwards = sensor measurements

timestamp_col = 0
label_col = 1

X = df.iloc[:, 2:].copy()
y = df.iloc[:, label_col].copy()


# ============================================================
# 3. HANDLE MISSING VALUES
# ============================================================

X = X.replace([np.inf, -np.inf], np.nan)

# Forward fill sensor values
X = X.ffill()

# Remove remaining NaNs
valid = ~X.isna().any(axis=1)

X = X.loc[valid]
y = y.loc[valid]


# ============================================================
# 4. REMOVE ACTIVITY 0 IF DESIRED
# ============================================================

# Activity 0 = transient / other
# Remove it if you want only actual activities.

mask = y != 0

X = X.loc[mask]
y = y.loc[mask]


# ============================================================
# 5. SCALE SENSOR FEATURES
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


print("Number of samples:", len(X_scaled))
print("Number of features:", X_scaled.shape[1])
print("Activities:", sorted(y.unique()))


# ============================================================
# 6. CREATE SEQUENCES USING SLIDING WINDOWS
# ============================================================

WINDOW_SIZE = 100
STEP = 50

sequences = []
sequence_labels = []

for start in range(0, len(X_scaled) - WINDOW_SIZE, STEP):

    end = start + WINDOW_SIZE

    window_X = X_scaled[start:end]
    window_y = y.iloc[start:end]

    # Only keep windows containing one activity
    if len(window_y.unique()) == 1:

        sequences.append(window_X)
        sequence_labels.append(window_y.iloc[0])


print("Number of sequences:", len(sequences))


# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

train_sequences, test_sequences, train_labels, test_labels = train_test_split(
    sequences,
    sequence_labels,
    test_size=0.2,
    random_state=42,
    stratify=sequence_labels
)


# ============================================================
# 8. TRAIN ONE HMM FOR EACH ACTIVITY
# ============================================================

activity_models = {}

N_STATES = 18

unique_activities = sorted(set(train_labels))

for activity in unique_activities:

    print(f"\nTraining HMM for activity {activity}")

    # Get all sequences belonging to this activity
    activity_sequences = [
        seq for seq, label in zip(train_sequences, train_labels)
        if label == activity
    ]

    if len(activity_sequences) == 0:
        continue

    # Combine sequences
    X_train = np.concatenate(activity_sequences)

    # Length of each sequence
    lengths = [len(seq) for seq in activity_sequences]

    model = GaussianHMM(
        n_components=N_STATES,
        covariance_type="diag",
        n_iter=50,
        random_state=42,
        verbose=True
    )

    model.fit(
        X_train,
        lengths=lengths
    )

    activity_models[activity] = model

    print(
        f"Activity {activity}: "
        f"{len(activity_sequences)} sequences trained"
    )


# ============================================================
# 9. CLASSIFY TEST SEQUENCES
# ============================================================

predictions = []

for seq in test_sequences:

    scores = {}

    for activity, model in activity_models.items():

        try:
            scores[activity] = model.score(seq)

        except Exception:
            scores[activity] = -np.inf

    predicted_activity = max(
        scores,
        key=scores.get
    )

    predictions.append(predicted_activity)


# ============================================================
# 10. EVALUATION
# ============================================================

accuracy = accuracy_score(
    test_labels,
    predictions
)

print("\n==============================")
print("HMM RESULTS")
print("==============================")

print("Accuracy:", accuracy)

print(
    classification_report(
        test_labels,
        predictions
    )
)

Number of samples: 249957
Number of features: 52
Activities: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(24)]
Number of sequences: 4972

Training HMM for activity 1


         1  404041.94166754             +nan
         2 4499292.72462100 +4095250.78295346
         3 5074334.70748575 +575041.98286475
         4 5284122.00329856 +209787.29581281
         5 5381379.10922081  +97257.10592225
         6 5415512.99034574  +34133.88112493
         7 5426420.66337136  +10907.67302562
         8 5430647.63547493   +4226.97210358
         9 5433372.23527116   +2724.59979623
        10 5434155.32695940    +783.09168824
        11 5436394.54948588   +2239.22252648
        12 5438704.99731623   +2310.44783035
        13 5440802.56847240   +2097.57115618
        14 5444767.60369131   +3965.03521890
        15 5446532.53088483   +1764.92719352
        16 5447549.58964650   +1017.05876167
        17 5448534.59294332    +985.00329682
        18 5449344.77269791    +810.17975458
        19 5449725.43827109    +380.66557319
        20 5449762.29733937     +36.85906828
        21 5449776.71201459     +14.41467522
        22 5449777.64812566      +0.93611107
        2

Activity 1: 434 sequences trained

Training HMM for activity 2


         1  815674.44041493             +nan
         2 4107340.35360650 +3291665.91319157
         3 4443048.06182578 +335707.70821928
         4 4618681.44447302 +175633.38264724
         5 4678533.02507490  +59851.58060187
         6 4693186.71814527  +14653.69307038
         7 4697819.72007876   +4633.00193348
         8 4704239.65478796   +6419.93470920
         9 4713497.66595392   +9258.01116596
C:\Users\Harin\AppData\Roaming\Python\Python310\site-packages\hmmlearn\hmm.py:352: RuntimeWarning: invalid value encountered in divide
  self.means_ = ((means_weight * means_prior + stats['obs'])
        10 4715603.67695671   +2106.01100279
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
        11              nan             +nan
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
        12              nan             +nan
Some rows of transmat_ have zero sum because no transition from the state was

Activity 2: 374 sequences trained

Training HMM for activity 3


...

         1    4652.75141522             +nan
         2 1567301.04962640 +1562648.29821118
         3 1684435.37523287 +117134.32560647
         4 1750594.15826221  +66158.78302933
         5 1784469.73631476  +33875.57805255
         6 1798902.07015385  +14432.33383909
C:\Users\Harin\AppData\Roaming\Python\Python310\site-packages\hmmlearn\hmm.py:352: RuntimeWarning: invalid value encountered in divide
  self.means_ = ((means_weight * means_prior + stats['obs'])
         7 1816048.97135279  +17146.90119894
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
         8              nan             +nan
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
         9              nan             +nan
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
        10              nan             +nan
Some rows of transmat_ have zero sum because no transition from the state 

Activity 16: 365 sequences trained

Training HMM for activity 17


         1 1103533.70184297             +nan
         2 2725371.01906709 +1621837.31722413
         3 2849100.30310035 +123729.28403326
         4 2899996.59897849  +50896.29587814
         5 2926461.48248531  +26464.88350681
         6 2945265.94418204  +18804.46169674
         7 2965752.15626099  +20486.21207895
C:\Users\Harin\AppData\Roaming\Python\Python310\site-packages\hmmlearn\hmm.py:352: RuntimeWarning: invalid value encountered in divide
  self.means_ = ((means_weight * means_prior + stats['obs'])
         8 2974806.18267924   +9054.02641824
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
         9              nan             +nan
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
        10              nan             +nan
Some rows of transmat_ have zero sum because no transition from the state was ever observed.
        11              nan             +nan
Some rows of transmat_ have z

Activity 24: 206 sequences trained

HMM RESULTS
Accuracy: 0.5648241206030151
              precision    recall  f1-score   support

           1       0.97      0.99      0.98       108
           2       0.00      0.00      0.00        94
           3       0.92      1.00      0.96        86
           4       1.00      1.00      1.00        89
           5       0.28      1.00      0.43        85
           6       0.00      0.00      0.00        94
           7       0.61      1.00      0.76        81
          12       0.30      1.00      0.46        63
          13       0.00      0.00      0.00        59
          16       0.00      0.00      0.00        91
          17       0.00      0.00      0.00        94
          24       1.00      1.00      1.00        51

    accuracy                           0.56       995
   macro avg       0.42      0.58      0.47       995
weighted avg       0.42      0.56      0.46       995



C:\Users\Harin\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Harin\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Harin\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i